In [5]:
# ============================================================
# 1. LOAD DATA
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    make_scorer,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE


# Path from experiments/ folder
DATA_PATH = "../data/processed/split"

# Load training and testing features
X_train = pd.read_csv(
    f"{DATA_PATH}/X_train_scaled.csv"
)

X_test = pd.read_csv(
    f"{DATA_PATH}/X_test_scaled.csv"
)

# Load target
y_train = pd.read_csv(
    f"{DATA_PATH}/y_train.csv"
).squeeze()

y_test = pd.read_csv(
    f"{DATA_PATH}/y_test.csv"
).squeeze()


# Check shapes
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (8000, 9)
X_test : (2000, 9)
y_train: (8000,)
y_test : (2000,)


In [6]:
# ============================================================
# 2. MLFLOW SETUP
# ============================================================

import mlflow
import mlflow.sklearn

mlflow.set_experiment("Predictive_Maintenance")

print("MLflow experiment set successfully.")

2026/09/05 16:38:46 INFO mlflow.tracking.fluent: Experiment with name 'Predictive_Maintenance' does not exist. Creating a new experiment.


MLflow experiment set successfully.


In [7]:
# ============================================================
# 3. LOGISTIC REGRESSION - NO BALANCING
# ============================================================

# 5-Fold Stratified Cross Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Logistic Regression model
lr_no_balance = LogisticRegression(
    solver="liblinear",
    max_iter=1000,
    random_state=42
)

# Store F1 scores from each fold
f1_scores_no_balance = []

with mlflow.start_run(
    run_name="Logistic_Regression_No_Balancing"
):

    for train_idx, val_idx in cv.split(X_train, y_train):

        # Split training data into training fold and validation fold
        X_train_fold = X_train.iloc[train_idx]
        X_val_fold = X_train.iloc[val_idx]

        y_train_fold = y_train.iloc[train_idx]
        y_val_fold = y_train.iloc[val_idx]

        # Train
        lr_no_balance.fit(
            X_train_fold,
            y_train_fold
        )

        # Predict validation fold
        y_val_pred = lr_no_balance.predict(
            X_val_fold
        )

        # Calculate F1
        fold_f1 = f1_score(
            y_val_fold,
            y_val_pred,
            zero_division=0
        )

        f1_scores_no_balance.append(fold_f1)

    # Mean and standard deviation
    cv_f1_mean_no_balance = np.mean(
        f1_scores_no_balance
    )

    cv_f1_std_no_balance = np.std(
        f1_scores_no_balance
    )

    # MLflow logging
    mlflow.log_param(
        "model",
        "Logistic Regression"
    )

    mlflow.log_param(
        "balancing_strategy",
        "No Balancing"
    )

    mlflow.log_param(
        "solver",
        "liblinear"
    )

    mlflow.log_param(
        "max_iter",
        1000
    )

    mlflow.log_metric(
        "cv_f1_mean",
        cv_f1_mean_no_balance
    )

    mlflow.log_metric(
        "cv_f1_std",
        cv_f1_std_no_balance
    )


# Display results
print("\n====================================")
print("LOGISTIC REGRESSION - NO BALANCING")
print("====================================")

print("Fold F1 Scores:")
print(f1_scores_no_balance)

print("\nMean CV F1:",
      cv_f1_mean_no_balance)

print("Std CV F1 :",
      cv_f1_std_no_balance)


LOGISTIC REGRESSION - NO BALANCING
Fold F1 Scores:
[0.4225352112676056, 0.4225352112676056, 0.3561643835616438, 0.3142857142857143, 0.3380281690140845]

Mean CV F1: 0.37070973787933076
Std CV F1 : 0.04435105883302326


In [8]:
# ============================================================
# 4. LOGISTIC REGRESSION - CLASS WEIGHT
# ============================================================

lr_class_weight = LogisticRegression(
    class_weight="balanced",
    solver="liblinear",
    max_iter=1000,
    random_state=42
)

f1_scores_class_weight = []

with mlflow.start_run(
    run_name="Logistic_Regression_Class_Weight"
):

    for train_idx, val_idx in cv.split(X_train, y_train):

        # Training and validation folds
        X_train_fold = X_train.iloc[train_idx]
        X_val_fold = X_train.iloc[val_idx]

        y_train_fold = y_train.iloc[train_idx]
        y_val_fold = y_train.iloc[val_idx]

        # Train
        lr_class_weight.fit(
            X_train_fold,
            y_train_fold
        )

        # Predict
        y_val_pred = lr_class_weight.predict(
            X_val_fold
        )

        # F1 score
        fold_f1 = f1_score(
            y_val_fold,
            y_val_pred,
            zero_division=0
        )

        f1_scores_class_weight.append(fold_f1)

    # Mean and standard deviation
    cv_f1_mean_class_weight = np.mean(
        f1_scores_class_weight
    )

    cv_f1_std_class_weight = np.std(
        f1_scores_class_weight
    )

    # MLflow logging
    mlflow.log_param(
        "model",
        "Logistic Regression"
    )

    mlflow.log_param(
        "balancing_strategy",
        "Class Weight"
    )

    mlflow.log_param(
        "class_weight",
        "balanced"
    )

    mlflow.log_param(
        "solver",
        "liblinear"
    )

    mlflow.log_param(
        "max_iter",
        1000
    )

    mlflow.log_metric(
        "cv_f1_mean",
        cv_f1_mean_class_weight
    )

    mlflow.log_metric(
        "cv_f1_std",
        cv_f1_std_class_weight
    )


print("\n====================================")
print("LOGISTIC REGRESSION - CLASS WEIGHT")
print("====================================")

print("Fold F1 Scores:")
print(f1_scores_class_weight)

print("\nMean CV F1:",
      cv_f1_mean_class_weight)

print("Std CV F1 :",
      cv_f1_std_class_weight)


LOGISTIC REGRESSION - CLASS WEIGHT
Fold F1 Scores:
[0.27945205479452057, 0.26625386996904027, 0.25862068965517243, 0.29102167182662536, 0.2674772036474164]

Mean CV F1: 0.27256509797855505
Std CV F1 : 0.011386308300730189


In [9]:
# ============================================================
# 5. LOGISTIC REGRESSION - SMOTE
# ============================================================

lr_smote = ImbPipeline([
    
    (
        "smote",
        SMOTE(
            random_state=42
        )
    ),
    
    (
        "model",
        LogisticRegression(
            solver="liblinear",
            max_iter=1000,
            random_state=42
        )
    )
])

f1_scores_smote = []

with mlflow.start_run(
    run_name="Logistic_Regression_SMOTE"
):

    for train_idx, val_idx in cv.split(X_train, y_train):

        # Training and validation folds
        X_train_fold = X_train.iloc[train_idx]
        X_val_fold = X_train.iloc[val_idx]

        y_train_fold = y_train.iloc[train_idx]
        y_val_fold = y_train.iloc[val_idx]

        # SMOTE + Logistic Regression
        lr_smote.fit(
            X_train_fold,
            y_train_fold
        )

        # Predict validation fold
        y_val_pred = lr_smote.predict(
            X_val_fold
        )

        # F1 score
        fold_f1 = f1_score(
            y_val_fold,
            y_val_pred,
            zero_division=0
        )

        f1_scores_smote.append(
            fold_f1
        )

    # Mean and standard deviation
    cv_f1_mean_smote = np.mean(
        f1_scores_smote
    )

    cv_f1_std_smote = np.std(
        f1_scores_smote
    )

    # MLflow logging
    mlflow.log_param(
        "model",
        "Logistic Regression"
    )

    mlflow.log_param(
        "balancing_strategy",
        "SMOTE"
    )

    mlflow.log_param(
        "solver",
        "liblinear"
    )

    mlflow.log_param(
        "max_iter",
        1000
    )

    mlflow.log_param(
        "smote_random_state",
        42
    )

    mlflow.log_metric(
        "cv_f1_mean",
        cv_f1_mean_smote
    )

    mlflow.log_metric(
        "cv_f1_std",
        cv_f1_std_smote
    )


print("\n====================================")
print("LOGISTIC REGRESSION - SMOTE")
print("====================================")

print("Fold F1 Scores:")
print(f1_scores_smote)

print("\nMean CV F1:",
      cv_f1_mean_smote)

print("Std CV F1 :",
      cv_f1_std_smote)


LOGISTIC REGRESSION - SMOTE
Fold F1 Scores:
[0.2801120448179272, 0.27184466019417475, 0.26299694189602446, 0.28, 0.2709677419354839]

Mean CV F1: 0.27318427776872206
Std CV F1 : 0.006402147207792164


In [10]:
# ============================================================
# 6. COMPARE IMBALANCE STRATEGIES
# ============================================================

strategy_results = {
    "No Balancing": (
        cv_f1_mean_no_balance,
        cv_f1_std_no_balance
    ),
    
    "Class Weight": (
        cv_f1_mean_class_weight,
        cv_f1_std_class_weight
    ),
    
    "SMOTE": (
        cv_f1_mean_smote,
        cv_f1_std_smote
    )
}


print("\n==============================================")
print("LOGISTIC REGRESSION - STRATEGY COMPARISON")
print("==============================================")

for strategy, (mean_f1, std_f1) in strategy_results.items():
    
    print(
        f"{strategy:<20} "
        f"Mean F1: {mean_f1:.4f}   "
        f"Std F1: {std_f1:.4f}"
    )


# Select strategy with highest Mean CV F1
best_strategy = max(
    strategy_results,
    key=lambda strategy: strategy_results[strategy][0]
)

best_cv_f1 = strategy_results[best_strategy][0]
best_cv_f1_std = strategy_results[best_strategy][1]


print("\n----------------------------------------------")
print("BEST IMBALANCE STRATEGY")
print("----------------------------------------------")
print("Strategy :", best_strategy)
print("Mean F1  :", round(best_cv_f1, 4))
print("Std F1   :", round(best_cv_f1_std, 4))


LOGISTIC REGRESSION - STRATEGY COMPARISON
No Balancing         Mean F1: 0.3707   Std F1: 0.0444
Class Weight         Mean F1: 0.2726   Std F1: 0.0114
SMOTE                Mean F1: 0.2732   Std F1: 0.0064

----------------------------------------------
BEST IMBALANCE STRATEGY
----------------------------------------------
Strategy : No Balancing
Mean F1  : 0.3707
Std F1   : 0.0444


In [12]:
# ============================================================
# 7. CREATE MODEL USING BEST STRATEGY
# ============================================================

if best_strategy == "No Balancing":

    base_model = LogisticRegression(
        solver="liblinear",
        max_iter=1000,
        random_state=42
    )

elif best_strategy == "Class Weight":

    base_model = LogisticRegression(
        class_weight="balanced",
        solver="liblinear",
        max_iter=1000,
        random_state=42
    )

elif best_strategy == "SMOTE":

    base_model = ImbPipeline([
        (
            "smote",
            SMOTE(random_state=42)
        ),
        (
            "model",
            LogisticRegression(
                solver="liblinear",
                max_iter=1000,
                random_state=42
            )
        )
    ])

print("Selected Strategy:", best_strategy)
print("\nBase Model:")
print(base_model)

Selected Strategy: No Balancing

Base Model:
LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')


In [13]:
# ============================================================
# 8. HYPERPARAMETER TUNING - GRIDSEARCHCV
# ============================================================

if best_strategy == "SMOTE":

    param_grid = {
        "model__C": [0.01, 0.1, 1, 10, 100],
        "model__penalty": ["l1", "l2"]
    }

else:

    param_grid = {
        "C": [0.01, 0.1, 1, 10, 100],
        "penalty": ["l1", "l2"]
    }


grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    return_train_score=False
)


with mlflow.start_run(
    run_name="Logistic_Regression_Hyperparameter_Tuning"
):

    grid_search.fit(
        X_train,
        y_train
    )

    # Best parameters
    best_params = grid_search.best_params_

    # Best CV F1
    best_grid_cv_f1 = grid_search.best_score_

    # MLflow logging
    mlflow.log_param(
        "model",
        "Logistic Regression"
    )

    mlflow.log_param(
        "balancing_strategy",
        best_strategy
    )

    for param, value in best_params.items():
        mlflow.log_param(
            param,
            value
        )

    mlflow.log_metric(
        "best_cv_f1",
        best_grid_cv_f1
    )


print("\n==============================================")
print("LOGISTIC REGRESSION - GRID SEARCH")
print("==============================================")

print("\nBest Parameters:")
print(best_params)

print("\nBest CV F1:")
print(best_grid_cv_f1)

/Users/sethukkarasipalani/Documents/AI4I-Predictive-Maintenance/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/sethukkarasipalani/Documents/AI4I-Predictive-Maintenance/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/sethukkarasipalani/Documents/AI4I-Predictive-Maintenance/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be


LOGISTIC REGRESSION - GRID SEARCH

Best Parameters:
{'C': 10, 'penalty': 'l1'}

Best CV F1:
0.4015040303461356


/Users/sethukkarasipalani/Documents/AI4I-Predictive-Maintenance/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/sethukkarasipalani/Documents/AI4I-Predictive-Maintenance/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [14]:
best_model = grid_search.best_estimator_

print("\n==============================================")
print("BEST LOGISTIC REGRESSION MODEL")
print("==============================================")

print(best_model)


BEST LOGISTIC REGRESSION MODEL
LogisticRegression(C=10, max_iter=1000, penalty='l1', random_state=42,
                   solver='liblinear')


In [15]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Test prediction completed.")

Test prediction completed.


In [16]:
accuracy = np.mean(y_pred == y_test)

precision = precision_score(
    y_test, y_pred, zero_division=0
)

recall = recall_score(
    y_test, y_pred, zero_division=0
)

f1 = f1_score(
    y_test, y_pred, zero_division=0
)

roc_auc = roc_auc_score(
    y_test, y_prob
)

average_precision = average_precision_score(
    y_test, y_prob
)

print("\n==============================================")
print("LOGISTIC REGRESSION - FINAL TEST EVALUATION")
print("==============================================")

print(f"Accuracy          : {accuracy:.4f}")
print(f"Precision         : {precision:.4f}")
print(f"Recall            : {recall:.4f}")
print(f"F1 Score          : {f1:.4f}")
print(f"ROC-AUC           : {roc_auc:.4f}")
print(f"Average Precision : {average_precision:.4f}")


LOGISTIC REGRESSION - FINAL TEST EVALUATION
Accuracy          : 0.9685
Precision         : 0.6087
Recall            : 0.2059
F1 Score          : 0.3077
ROC-AUC           : 0.9312
Average Precision : 0.4616


In [17]:
cm = confusion_matrix(y_test, y_pred)

print("\n==============================================")
print("LOGISTIC REGRESSION - CONFUSION MATRIX")
print("==============================================")

print(cm)


LOGISTIC REGRESSION - CONFUSION MATRIX
[[1923    9]
 [  54   14]]


In [18]:
print("\n==============================================")
print("LOGISTIC REGRESSION - CLASSIFICATION REPORT")
print("==============================================")

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)


LOGISTIC REGRESSION - CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      1932
           1       0.61      0.21      0.31        68

    accuracy                           0.97      2000
   macro avg       0.79      0.60      0.65      2000
weighted avg       0.96      0.97      0.96      2000



In [19]:
with mlflow.start_run(
    run_name="Logistic_Regression_Final_Evaluation"
):
    
    # Model parameters
    mlflow.log_param("model", "Logistic Regression")
    mlflow.log_param("balancing_strategy", best_strategy)
    
    # Best hyperparameters
    for param, value in best_params.items():
        mlflow.log_param(param, value)
    
    # Cross-validation performance
    mlflow.log_metric("cv_f1_mean", best_grid_cv_f1)
    
    # Test performance
    mlflow.log_metric("test_accuracy", accuracy)
    mlflow.log_metric("test_precision", precision)
    mlflow.log_metric("test_recall", recall)
    mlflow.log_metric("test_f1", f1)
    mlflow.log_metric("test_roc_auc", roc_auc)
    mlflow.log_metric("test_average_precision", average_precision)
    
    # Log final model
    mlflow.sklearn.log_model(
        sk_model=best_model,
        name="logistic_regression_model"
    )

print("\n==============================================")
print("FINAL LOGISTIC REGRESSION MLflow RUN")
print("==============================================")
print("Model and metrics logged successfully.")


FINAL LOGISTIC REGRESSION MLflow RUN
Model and metrics logged successfully.


In [21]:
print("\n==============================================")
print("MLFLOW - PREDICTIVE MAINTENANCE RUNS")
print("==============================================")

runs = mlflow.search_runs(
    experiment_names=["Predictive_Maintenance"]
)

print("Available columns:")
print(runs.columns.tolist())

print("\nNumber of runs:", len(runs))

display(runs)


MLFLOW - PREDICTIVE MAINTENANCE RUNS
Available columns:
['run_id', 'experiment_id', 'status', 'artifact_uri', 'start_time', 'end_time', 'metrics.test_average_precision', 'metrics.test_accuracy', 'metrics.test_recall', 'metrics.test_f1', 'metrics.test_precision', 'metrics.cv_f1_mean', 'metrics.test_roc_auc', 'metrics.best_cv_f1', 'metrics.cv_f1_std', 'params.penalty', 'params.C', 'params.model', 'params.balancing_strategy', 'params.solver', 'params.max_iter', 'params.smote_random_state', 'params.class_weight', 'tags.mlflow.source.type', 'tags.mlflow.user', 'tags.mlflow.source.name', 'tags.mlflow.runName']

Number of runs: 5


,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.test_average_precision,metrics.test_accuracy,metrics.test_recall,metrics.test_f1,...,params.model,params.balancing_strategy,params.solver,params.max_iter,params.smote_random_state,params.class_weight,tags.mlflow.source.type,tags.mlflow.user,tags.mlflow.source.name,tags.mlflow.runName
0,51350045d9e74251b3aae6cf1789070c,1,FINISHED,/Users/sethukkarasipalani/Documents/AI4I-Predi...,2026-09-05 11:14:48.711000+00:00,2026-09-05 11:14:53.062000+00:00,0.461619,0.9685,0.205882,0.307692,...,Logistic Regression,No Balancing,NaN,NaN,NaN,NaN,NOTEBOOK,sethukkarasipalani,5_model_eexperiments.ipynb,Logistic_Regression_Final_Evaluation
1,b71dd53da5fa4967ac9a9ab5f43ce638,1,FINISHED,/Users/sethukkarasipalani/Documents/AI4I-Predi...,2026-09-05 11:12:08.121000+00:00,2026-09-05 11:12:10.830000+00:00,NaN,NaN,NaN,NaN,...,Logistic Regression,No Balancing,NaN,NaN,NaN,NaN,NOTEBOOK,sethukkarasipalani,5_model_eexperiments.ipynb,Logistic_Regression_Hyperparameter_Tuning
2,928a8a45b9ef42f1b0ed94d6b8f01dd6,1,FINISHED,/Users/sethukkarasipalani/Documents/AI4I-Predi...,2026-09-05 11:10:07.894000+00:00,2026-09-05 11:10:08.034000+00:00,NaN,NaN,NaN,NaN,...,Logistic Regression,SMOTE,liblinear,1000,42,NaN,NOTEBOOK,sethukkarasipalani,5_model_eexperiments.ipynb,Logistic_Regression_SMOTE
3,483405e062bf44178648bbad0ff9c886,1,FINISHED,/Users/sethukkarasipalani/Documents/AI4I-Predi...,2026-09-05 11:09:41.751000+00:00,2026-09-05 11:09:41.841000+00:00,NaN,NaN,NaN,NaN,...,Logistic Regression,Class Weight,liblinear,1000,NaN,balanced,NOTEBOOK,sethukkarasipalani,5_model_eexperiments.ipynb,Logistic_Regression_Class_Weight
4,7789652d6dba4322b031bc1337236c11,1,FINISHED,/Users/sethukkarasipalani/Documents/AI4I-Predi...,2026-09-05 11:09:12.601000+00:00,2026-09-05 11:09:12.697000+00:00,NaN,NaN,NaN,NaN,...,Logistic Regression,No Balancing,liblinear,1000,NaN,NaN,NOTEBOOK,sethukkarasipalani,5_model_eexperiments.ipynb,Logistic_Regression_No_Balancing
